# 2CSSID Lab02. Naïve Bayes

<p style='text-align: right;font-style: italic; color:red;'>Designed by: Mr. Abdelkrime Aries</p>
<p style='text-align: right;font-style: italic; color:blue;'>Version 20261024</p>

In this lab, we will learn about Naive Bayes by testing 2 implementations:
- Multinomial Naïve Bayes
- Gaussian Naïve Bayes

**Team:**
- **Member 01**: Rafa Houssam
- **Member 02**: Dali Brahammohammed
- **Group**: 01


In [3]:
import sys, timeit
from typing          import Tuple, List, Type
from collections.abc import Callable

sys.version

'3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]'

In [4]:
import numpy             as np

import pandas            as pd 
import matplotlib.pyplot as plt 
import matplotlib
%matplotlib inline

np.__version__, pd.__version__, matplotlib.__version__

('2.5.3', '3.0.6', '3.11.2')

In [5]:
import sklearn

from sklearn.naive_bayes   import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics       import classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection         import train_test_split
from sklearn.naive_bayes             import MultinomialNB, GaussianNB
from sklearn.linear_model            import LogisticRegression
from sklearn.tree                    import DecisionTreeClassifier
from sklearn.metrics                 import precision_score, recall_score
import timeit


sklearn.__version__

'1.9.1'

## I. Algorithms implementation

In this section, we will try to implement multinomial Naive Bayes.


**>> Try to use "numpy" which will save a lot of time and effort**

In [6]:
# Dataset play 

# outlook & temperature & humidity & windy
Xplay = np.array([
    ['sunny'   , 'hot' , 'high'  , 'no'],
    ['sunny'   , 'hot' , 'high'  , 'yes'],
    ['overcast', 'hot' , 'high'  , 'no'],
    ['rainy'   , 'mild', 'high'  , 'no'],
    ['rainy'   , 'cool', 'normal', 'no'],
    ['rainy'   , 'cool', 'normal', 'yes'],
    ['overcast', 'cool', 'normal', 'yes'],
    ['sunny'   , 'mild', 'high'  , 'no'],
    ['sunny'   , 'cool', 'normal', 'no'],
    ['rainy'   , 'mild', 'normal', 'no'],
    ['sunny'   , 'mild', 'normal', 'yes'],
    ['overcast', 'mild', 'high'  , 'yes'],
    ['overcast', 'hot' , 'normal', 'no'],
    ['rainy'   , 'mild', 'high'  , 'yes']
])

Yplay = np.array([
    'no', 
    'no', 
    'yes', 
    'yes', 
    'yes', 
    'no', 
    'yes', 
    'no', 
    'yes', 
    'yes', 
    'yes', 
    'yes', 
    'yes', 
    'no'
])

len(Xplay), len(Yplay)

(14, 14)

In [7]:
# height & weight & footsize & person
Xperson = np.array([
    [182., 81.6, 30.],
    [180., 86.2, 28.],
    [170., 77.1, 30.],
    [180., 74.8, 25.],
    [152., 45.4, 15.],
    [168., 68.0, 20.],
    [165., 59.0, 18.],
    [175., 68.0, 23.]
])

Yperson = np.array([
    'male', 'male', 'male', 'male',
    'female', 'female', 'female', 'female'
])

len(Xperson), len(Yperson)

(8, 8)

### I.1. Prior statistics
L

Given an output list $Y[M]$, the probability of each class $c$ is estimated as:
$$p(c) = \frac{\#(Y = c)}{|Y|}$$

In here, we want to store the frequencies of different classes.
Our function must return two lists:
- One containing the names of unique classes.
- Another containing their frequencies.

In [ ]:
# L*(N+1) OVR
# (N+1)L(L-1)/2

In [8]:
# Prior statistics
def fit_prior(Y: 'np.ndarray[M](str)') -> Tuple['np.ndarray[K](str)', 'np.ndarray[K](int)']: 
    return np.unique(Y, return_counts=True)


((array(['no', 'yes'], dtype='<U3'), array([5, 9])),
 (array(['female', 'male'], dtype='<U6'), array([4, 4])))

### I.2. Multinomial Law

In this section, we will implement multinomial naive Bayes from scratch using Numpy.

#### I.2.1. Multinomial Likelihood statistics

Given:
- $A$: a categorical feature
- $Y$: the ouput
- $C$: the classes

The function takes as argument $A, Y, C$ previously described.
It must return:
- $V$: unique values of this feature (feature's categories)
- $S[|C|, |V|]$: a matrix containing count $\#(Y = c \wedge A = v),\, \forall c \in C, \forall v \in A$

In [ ]:
# Multinomial Likelihood statistics
def fit_multinomial_likelihood(A: 'np.ndarray[M](str)', 
                               Y: 'np.ndarray[M](str)', 
                               C: 'np.ndarray[C](str)'
                               ) -> Tuple['np.ndarray[V](str)', 'np.ndarray[C, V](int)']: 

    V = np.unique(A)
    S = np.zeros((len(C), len(V)), dtype=int)
    for c_idx, c in enumerate(C):
        for v_idx, v in enumerate(V):
            S[c_idx, v_idx] = np.sum((Y == c) & (A == v))
    return V, S


#### I.2.2. Multinomial Likelihood training

**Nothing to code here, although you have to know how it functions for next use**

This function aims to generate parameters $\theta$. 
In our case, paramters are diffrent from those of *logistic regrssion*.
They are a dictionary (map) with two entries:
- "prior": a dictionary having "vocab" a list of values and "freq" a list of their respective frequencies.
- "likelihood": a list of dictionaries representing statistics of each feature (the same order of $X$ features)

In [ ]:
def fit_multinomial_NB(X: 'np.ndarray[M, N](str)', 
                       Y: 'np.ndarray[M](str)'
                       ) -> object: 
    
    Theta   = {'prior': {}, 'likelihood': []}

    Theta['prior']['vocab'], Theta['prior']['freq'] = fit_prior(Y)

    for j in range(X.shape[1]): 
        likelihood = {}
        likelihood['vocab'], likelihood['freq'] = fit_multinomial_likelihood(X[:, j], Y, Theta['prior']['vocab'])
        Theta['likelihood'].append(likelihood)
    
    return Theta


#=====================================================================
# UNIT TEST
#=====================================================================
# Result: 
# {'prior': {'vocab': array(['no', 'yes'], dtype='<U3'), 'freq': array([5, 9])},
#  'likelihood': [{'vocab': array(['overcast', 'rainy', 'sunny'], dtype='<U8'),
#    'freq': array([[0, 2, 3],
#           [4, 3, 2]])},
#   {'vocab': array(['cool', 'hot', 'mild'], dtype='<U8'),
#    'freq': array([[1, 2, 2],
#           [3, 2, 4]])},
#   {'vocab': array(['high', 'normal'], dtype='<U8'),
#    'freq': array([[4, 1],
#           [3, 6]])},
#   {'vocab': array(['no', 'yes'], dtype='<U8'),
#    'freq': array([[2, 3],
#           [6, 3]])}]}
#---------------------------------------------------------------------
Theta_play = fit_multinomial_NB(Xplay, Yplay)

Theta_play

#### I.2.3. Multinomial Likelihood prediction

Given:
- $A$: a categorical feature
- $V$: unique values of this feature (feature's categories)
- $Y$: the ouput
- $C$: the classes
- $\alpha$: smoothing factor

Log likelihood is calculated as:
$$ \log p(A=v|Y=c) = \log(\#(Y = k \wedge A = v) + \alpha) - \log(\#(y = k) + \alpha * |V|)$$


In [ ]:
# You can use this function in the next implimentation
# It takes a list of unique values V and a given value v
# It returns the position of v in V
# If v does not exist in V, it rturns -1
def find_idx(V: np.ndarray, v: str) -> int:
    k = np.argwhere(V == v).flatten()
    if len(k):
        return k[0]
    return -1

V_t = np.array(['One', 'Two', 'Three'])
find_idx(V_t, 'Two'), find_idx(V_t, 'Four')

In [ ]:
# Multinomial Likelihood prediction
def predict_multinomial_NB1(v: str, 
                            j: int,
                            Theta: object,  
                            alpha: float = 0.
                            ) -> 'np.ndarray[C](float)':

    likelihood = Theta['likelihood'][j]
    vocab = likelihood['vocab']
    value_idx = find_idx(vocab, v)
    if value_idx == -1:
        counts = np.zeros(len(Theta['prior']['vocab']), dtype=float)
    else:
        counts = likelihood['freq'][:, value_idx].astype(float)

    class_counts = Theta['prior']['freq']
    denominator = class_counts + alpha * len(vocab)
    return np.log(counts + alpha) - np.log(denominator)


### I.3. Normal (Gaussian) Law

In this section, we will implement gaussian naive Bayes from scratch using Numpy.

#### I.3.1. Gaussian Likelihood statistics

Given:
- $A$: a categorical feature
- $Y$: the ouput
- $C$: the classes

The function takes as argument $A, Y, C$ previously described.
It must return $S[|C|, 2, N]$; a tensor having these dimensions:
- first dimension: each element represents one class's statistics
- second dimension: 1st element represents means; 2ns element represents variances
- third dimension: each element represents mean/variance of the respective feature

In [ ]:
# Gaussian Likelihood statistics
def fit_gaussian_likelihood(X: 'np.ndarray[M](float)', 
                            Y: 'np.ndarray[M](str)', 
                            C: 'np.ndarray[C](str)'
                            ) -> Tuple['np.ndarray[C, 2, N](float)']: 
    stats = np.zeros((len(C), 2, X.shape[1]), dtype=float)
    for c_idx, c in enumerate(C):
        X_class = X[Y == c]
        stats[c_idx, 0, :] = np.mean(X_class, axis=0)
        stats[c_idx, 1, :] = np.var(X_class, axis=0)
    return stats


#### I.3.2. Gaussian Likelihood training

**Nothing to code here, although you have to know how it functions for next use**

This function aims to generate parameters $\theta$. 
In our case, paramters are diffrent from those of *logistic regrssion*.
They are a dictionary (map) with two entries:
- "prior": a dictionary having "vocab" a list of values and "freq" a list of their respective frequencies.
- "likelihood": a tensor of shape $[|C|, 2, N]$ containing likelihood statistics

In [ ]:
def fit_gaussian_NB(X: 'np.ndarray[M, N](str)', 
                    Y: 'np.ndarray[M](str)'
                    ) -> object: 
    
    Theta   = {'prior': {}, 'likelihood': []}

    Theta['prior']['vocab'], Theta['prior']['freq'] = fit_prior(Y)
    Theta['likelihood'] = fit_gaussian_likelihood(X, Y, Theta['prior']['vocab'])

    return Theta


#=====================================================================
# UNIT TEST
#=====================================================================
# Result: 
# {'prior': {'vocab': array(['female', 'male'], dtype='<U6'),
#   'freq': array([4, 4])},
#  'likelihood': array([[[165.        ,  60.1       ,  19.        ],
#          [ 92.66666667, 114.04      ,  11.33333333]],
 
#         [[178.        ,  79.925     ,  28.25      ],
#          [ 29.33333333,  25.47583333,   5.58333333]]])}
#---------------------------------------------------------------------
Theta_person = fit_gaussian_NB(Xperson, Yperson)

Theta_person

#### I.2.4. Gaussian Likelihood prediction

Given:
- $A$: a numerical feature
- $\mu_{Ac}$: mean of values of feature $A$ having $c$ as class
- $\sigma_{Ac}$: variance of values of feature $A$ having $c$ as class
- $Y$: the output
- $C$: the classes

Log likelihood is calculated as:
$$ \log p(A=v|Y=c) = \frac{-(v-\mu_{Ac})^2}{2 \sigma_{Ac}^2} - \log(\sqrt{2\pi \sigma_{Ac}^2})$$

In [ ]:
# Gaussian Likelihood prediction
def predict_gaussian_NB1(v: str, 
                         j: int,
                         Theta: object,  
                         alpha: float = 0. # this is just added for compatibility
                         ) -> 'np.ndarray[C](float)':

    means = Theta['likelihood'][:, 0, j]
    variances = Theta['likelihood'][:, 1, j]
    return -((v - means) ** 2) / (2 * variances) - np.log(np.sqrt(2 * np.pi * variances))


### I.4. Final prediction

Our goal is to calculate approximate log probabilities of all classes given a sample:
$$\log P(y=c_k | \overrightarrow{x} = \overrightarrow{f})  \approx \log P(y=c_k) + \sum\limits_{f_j \in \overrightarrow{f}} \log P(f_j = x_j|y=c_k)$$

This function takes:
- $X^{(i)}$ one sample with $N$ features
- $\theta$ parameters (either those of multinomial or gaussian)
- $pred_{fct}$ a function to predict one feauture (either multinomial or gaussian)
- add_prior: if True, add prior probability
- $\alpha$ smoothing factor (passing it to gaussian function will do nothing)

It must return a vector of probabilities

In [ ]:
# Final prediction
def predict_NB1(Xi    : 'np.ndarray[N]', 
                Theta: object,  
                pred_fct: Callable,
                add_prior: bool  = True,
                alpha: float = 1.0
                ) -> 'np.ndarray[C](float)':

    class_freq = Theta['prior']['freq']
    if add_prior:
        log_p = np.log(class_freq) - np.log(class_freq.sum())
    else:
        log_p = np.zeros(len(class_freq), dtype=float)

    for j, value in enumerate(Xi):
        log_p += pred_fct(value, j, Theta, alpha=alpha)

    return log_p


### I.5. Final product

**>> Nothing to code here**


In [ ]:
class NaiveBayes(object): 

    def __init__(self, multinomial=True):
        if multinomial:
            self.train = fit_multinomial_NB
            self.pred = predict_multinomial_NB1
        else:
            self.train = fit_gaussian_NB
            self.pred = predict_gaussian_NB1
        
    def fit(self, X, Y):
        self.Theta = self.train(X, Y)
    
    def predict(self, X, add_prior=True, prob=False, alpha=0.): 
        Y_pred = []
        for i in range(len(X)): 
            Y_pred.append(predict_NB1(
                X[i,:], self.Theta, self.pred, add_prior=add_prior, alpha=alpha
                ))
        
        Y_pred = np.array(Y_pred)

        if prob:
            return Y_pred

        return np.choose(np.argmax(Y_pred, axis=1), self.Theta['prior']['vocab'])


#=====================================================================
# UNIT TEST
#=====================================================================
# Result: 
# (array(['yes', 'yes', 'no'], dtype='<U3'),
#  array([[-5.20912179, -4.10264337],
#         [-6.30773408, -5.48893773],
#         [-3.88736595, -4.67800751]]),
#  array(['female', 'male'], dtype='<U6'),
#  array([[-11.09424018, -22.73291741],
#         [-15.27968966, -12.41764665]]))
#---------------------------------------------------------------------

multinomial_nb = NaiveBayes()
multinomial_nb.fit(Xplay, Yplay)

gaussian_nb = NaiveBayes(multinomial=False)
gaussian_nb.fit(Xperson, Yperson)

X_t1 = np.array([
    ['rainy', 'cool', 'normal', 'yes'],
    ['snowy', 'cool', 'normal', 'yes'],
    ['sunny', 'hot' , 'high', 'no']
])

X_t2 = np.array([
    [183., 59., 20.],
    [175., 65., 30.]
])


multinomial_nb.predict(X_t1, alpha=1.), \
    multinomial_nb.predict(X_t1, alpha=1., prob=True), \
    gaussian_nb.predict(X_t2), \
    gaussian_nb.predict(X_t2, prob=True)


## II. Application and Analysis

In this section, we will test different concepts by running an experiment, formulating a hypothesis and trying to justify it.

### II.1. Prior probability 

We want to test the effect of prior probability.
To do this, we trained two models:
1. With prior probability
1. Without prior probability (It considers a uniform distribution of classes)

To test whether the models have adapted well to the training dataset, we will test them on the same dataset and calculate the classification ratio.


In [16]:
nb_withPrior     = CategoricalNB(alpha=1.0, fit_prior=True )
nb_noPrior       = CategoricalNB(alpha=1.0, fit_prior=False)

enc         = OrdinalEncoder()
Xplay_tf    = enc.fit_transform(Xplay)
nb_withPrior.fit(Xplay_tf, Yplay)
nb_noPrior.fit(Xplay_tf, Yplay)

Ypred_withPrior = nb_withPrior.predict(Xplay_tf)
Ypred_noPrior = nb_noPrior.predict(Xplay_tf)


print( 'Considring prior probability'  )
print(classification_report(Yplay, Ypred_withPrior))

print( 'No prior probability'  )
print(classification_report(Yplay, Ypred_noPrior))

Considring prior probability
              precision    recall  f1-score   support

          no       1.00      0.80      0.89         5
         yes       0.90      1.00      0.95         9

    accuracy                           0.93        14
   macro avg       0.95      0.90      0.92        14
weighted avg       0.94      0.93      0.93        14

No prior probability
              precision    recall  f1-score   support

          no       0.67      0.80      0.73         5
         yes       0.88      0.78      0.82         9

    accuracy                           0.79        14
   macro avg       0.77      0.79      0.78        14
weighted avg       0.80      0.79      0.79        14



**Analysis**

1. With the prior, training accuracy is about 0.93; without it, accuracy is about 0.79. Since the class counts are unequal (5 `no` and 9 `yes`), the prior is useful for this example and improves the reported fit on this dataset.
2. The classifier adds each class's log prior to its feature log-likelihood. A more common class starts with a higher score, so the prior can change which class wins when feature evidence is close. Here it shifts the predictions toward the more frequent `yes` class and improves the measured accuracy.
3. If the classes are equally likely, their log priors are the same constant. That constant does not change the argmax prediction, so using the prior is unnecessary for classification when a uniform prior is appropriate.


### II.2. Smoothing

We want to test the Lidstone smoothing's effect.
To do this, we trained three models:
1. alpha = 1 (Laplace smoothing)
1. alpha = 0.5
1. alpha = 0 (without smoothing)

In [17]:
NBC_10 = CategoricalNB(alpha = 1.0 )
NBC_05 = CategoricalNB(alpha = 0.5 )
NBC_00 = CategoricalNB(alpha = 0.0 )

NBC_10.fit( Xplay_tf,   Yplay )
NBC_05.fit( Xplay_tf,   Yplay )
NBC_00.fit( Xplay_tf,   Yplay )

Y_10   = NBC_10.predict(Xplay_tf)
Y_05   = NBC_05.predict(Xplay_tf)
Y_00   = NBC_00.predict(Xplay_tf)


print(                'Alpha = 1.0'                        )
print(classification_report(Yplay, Y_10, zero_division=0.0))

print(                'Alpha = 0.5'                        )
print(classification_report(Yplay, Y_05, zero_division=0.0))

print(                'Alpha = 0.0'                        )
print(classification_report(Yplay, Y_00, zero_division=0.0))


Alpha = 1.0
              precision    recall  f1-score   support

          no       1.00      0.80      0.89         5
         yes       0.90      1.00      0.95         9

    accuracy                           0.93        14
   macro avg       0.95      0.90      0.92        14
weighted avg       0.94      0.93      0.93        14

Alpha = 0.5
              precision    recall  f1-score   support

          no       1.00      0.80      0.89         5
         yes       0.90      1.00      0.95         9

    accuracy                           0.93        14
   macro avg       0.95      0.90      0.92        14
weighted avg       0.94      0.93      0.93        14

Alpha = 0.0
              precision    recall  f1-score   support

          no       1.00      0.80      0.89         5
         yes       0.90      1.00      0.95         9

    accuracy                           0.93        14
   macro avg       0.95      0.90      0.92        14
weighted avg       0.94      0.93     

/opt/env/ml/lib/python3.10/site-packages/sklearn/naive_bayes.py:1529: RuntimeWarning: divide by zero encountered in log
  np.log(smoothed_cat_count) - np.log(smoothed_class_count.reshape(-1, 1))


**Analysis**

1. The classification reports are identical for alpha=1.0, 0.5, and 0.0: accuracy remains about 0.93, with the same precision, recall, and F1 scores. Smoothing changes the estimated likelihoods but does not change the predicted labels in this dataset.
2. The examples are classified the same way because the likelihood score rankings remain unchanged for these three alpha values. Smoothing affects the probabilities, but on this small training set the adjustments are not large enough to change the winning class for any example.
3. With alpha=0, a category that never appeared with a class has a count of zero. Taking its logarithm requires log(0), which is undefined and produces negative infinity and a divide-by-zero runtime warning.
4. Smoothing gives unseen feature/class combinations a nonzero estimated probability. This prevents a single unseen feature from making the entire class probability zero and generally makes predictions more robust on new data.


### II.3. Naive Bayes performance

Naive Bayes is known to generate powerful models when it comes to classifying textual documents.
We want to test this proposition using spam detection over [SMS Spam Collection Dataset](https://www.kaggle.com/uciml/sms-spam-collection-dataset) dataset.

Each message is represented using term frequency (TF), where a word is considered as a feature.
In this case, a message is represented by a vector of frequencies (how many times each word appeared in the message).
We want to compare these models:
1. Multinomial Naive Bayes (MNB)
1. Gaussian Naive Bayes (GNB)
1. Logistic Regression (LR) 

In [18]:
# reading the dataset
messages = pd.read_csv('data/spam.csv', encoding='latin-1')
# renaming features: text and class
messages = messages.rename(columns={'v1': 'class', 'v2': 'text'})
# keeping only these two features
messages = messages.filter(['text', 'class'])

messages.head()

,text,class
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


In [19]:
models = [
    MultinomialNB(),
    GaussianNB(),
    LogisticRegression(solver='lbfgs'),
    #solver=sag is slower; so I chose the fastest
]

algos = [
    'Multinomial Naive Bayes (MNB)', 
    'Gaussian Naive Bayes  (GNB)', 
    'Logistic Regression (LR)', 
]

perf = {
    'train_time': [],
    'test_time' : [],
    'recall'    : [],
    'precision' : []
}


msg_train, msg_test, Y_train, Y_test = train_test_split(messages['text'] ,
                                                        messages['class'],
                                                        test_size    = 0.2, 
                                                        random_state = 0  )

count_vectorizer = CountVectorizer()
X_train          = count_vectorizer.fit_transform(msg_train).toarray()
X_test           = count_vectorizer.transform    (msg_test ).toarray()


for model in models:
    # ==================================
    # TRAIN 
    # ==================================
    start_time = timeit.default_timer()
    model.fit(X_train, Y_train)
    perf['train_time'].append(timeit.default_timer() - start_time)
    
    # ==================================
    # TEST 
    # ==================================
    start_time = timeit.default_timer()
    Y_pred     = model.predict(X_test)
    perf['test_time'].append(timeit.default_timer() - start_time)
    
    # ==================================
    # PERFORMANCE 
    # ==================================
    # In here, we are interrested in "spam" class which is our positive class
    perf['precision'].append(precision_score(Y_test, Y_pred, pos_label='spam'))
    perf['recall'   ].append(recall_score   (Y_test, Y_pred, pos_label='spam'))

    
pd.DataFrame({
    'Algorithm' : algos,
    'Train time': perf['train_time'],
    'Test time' : perf['test_time'],
    'Precision' : perf['precision'],
    'Recall'    : perf['recall']
})

,Algorithm,Train time,Test time,Precision,Recall
0,Multinomial Naive Bayes (MNB),0.260616,0.018093,0.987179,0.927711
1,Gaussian Naive Bayes (GNB),0.325610,0.073260,0.616667,0.891566
2,Logistic Regression (LR),0.517445,0.016002,0.986111,0.855422


**Analysis**

- Naive Bayes (MNB, GNB) vs. Logistic Regression (LR)
    1. In the saved results, training is fastest for MNB (about 0.261 s), then GNB (0.326 s), then LR (0.517 s). Testing is fastest for LR (0.016 s), then MNB (0.018 s), then GNB (0.073 s). These are timings from this run and can vary across runs and machines.
    2. MNB mostly accumulates word/class counts and normalizes them, so its training is inexpensive. GNB computes per-class means and variances for each feature, which takes more work. LR iteratively optimizes its weights over the training examples and vocabulary, so it takes the longest to train here.
    3. LR is slightly faster to test than MNB in this run, while GNB is much slower than both.
    4. LR prediction is a matrix/vector score calculation followed by a class decision. MNB scores each class from the word features using stored likelihoods. GNB evaluates a Gaussian density for each feature and class, which involves more arithmetic per feature; that explains its longer test time here. The LR/MNB difference is small and implementation-dependent.
- MNB vs. GNB
    1. MNB uses discrete feature counts and class/word count tables for the multinomial model. GNB must estimate means and variances and evaluate Gaussian densities for its features, adding arithmetic and storage per class and feature.
    2. SMS messages are represented as nonnegative word counts. This sparse, discrete bag-of-words representation fits the multinomial model's assumptions. GaussianNB treats each word-count feature as continuous and Gaussian, which poorly matches the many zeros and skewed count distributions; the notebook also converts the vectors to dense arrays, adding cost.
    3. Storing raw counts makes training and incremental updates straightforward, but prediction must normalize counts and take logarithms unless those log-likelihoods are precomputed. Storing precomputed log probabilities makes prediction faster, at the cost of computing and storing them during training; counts are still needed to update the model correctly.
    4. A count-based MNB is convenient for new SMS data: update the class and word counts with the new examples, then recompute the likelihoods. GNB can also be updated if it keeps sufficient statistics such as counts, sums, and sums of squares, but means and variances alone are not enough to merge new data exactly.


In [20]:
print("  _____    __                                              _               ")
print(" |_   _|  / _|                                            | |              ")
print("   | |   | |_     _   _    ___    _   _      __ _    ___  | |_             ")
print("   | |   |  _|   | | | |  / _ \  | | | |    / _` |  / _ \ | __|            ")
print("  _| |_  | |     | |_| | | (_) | | |_| |   | (_| | |  __/ | |_             ")
print(" |_____| |_|      \__, |  \___/   \__,_|    \__, |  \___|  \__|            ")
print("                   __/ |                     __/ |                         ")
print("                  |___/                     |___/                          ")
print("  _     _       _            __                                            ")
print(" | |   | |     (_)          / _|                 _                         ")
print(" | |_  | |__    _   ___    | |_    __ _   _ __  (_)                        ")
print(" | __| | '_ \  | | / __|   |  _|  / _` | | '__|                            ")
print(" | |_  | | | | | | \__ \   | |   | (_| | | |     _                         ")
print("  \__| |_| |_| |_| |___/   |_|    \__,_| |_|    ( )                        ")
print("                                                |/                         ")
print("                                                                           ")
print("                                                                           ")
print("                                                                           ")
print("  _   _    ___    _   _      __ _   _ __    ___                            ")
print(" | | | |  / _ \  | | | |    / _` | | '__|  / _ \                           ")
print(" | |_| | | (_) | | |_| |   | (_| | | |    |  __/                           ")
print("  \__, |  \___/   \__,_|    \__,_| |_|     \___|                           ")
print("   __/ |                                                                   ")
print("  |___/                                                                    ")
print("                    _                                                __    ")
print("                   | |                                            _  \ \   ")
print("  _ __     ___     | |__    _   _   _ __ ___     __ _   _ __     (_)  | |  ")
print(" | '_ \   / _ \    | '_ \  | | | | | '_ ` _ \   / _` | | '_ \         | |  ")
print(" | | | | | (_) |   | | | | | |_| | | | | | | | | (_| | | | | |    _   | |  ")
print(" |_| |_|  \___/    |_| |_|  \__,_| |_| |_| |_|  \__,_| |_| |_|   (_)  | |  ")
print("                                                                     /_/   ")
print("                                                                           ")

  _____    __                                              _               
 |_   _|  / _|                                            | |              
   | |   | |_     _   _    ___    _   _      __ _    ___  | |_             
   | |   |  _|   | | | |  / _ \  | | | |    / _` |  / _ \ | __|            
  _| |_  | |     | |_| | | (_) | | |_| |   | (_| | |  __/ | |_             
 |_____| |_|      \__, |  \___/   \__,_|    \__, |  \___|  \__|            
                   __/ |                     __/ |                         
                  |___/                     |___/                          
  _     _       _            __                                            
 | |   | |     (_)          / _|                 _                         
 | |_  | |__    _   ___    | |_    __ _   _ __  (_)                        
 | __| | '_ \  | | / __|   |  _|  / _` | | '__|                            
 | |_  | | | | | | \__ \   | |   | (_| | | |     _                         
  \__| |_| |